# BearID PyTorch Distributed Deep Metric Learning Pipeline

This pipeline is fully decoupled. Components use `@dsl.container_component` executing pre-baked Python scripts from the GHCR images.
Hardware resources are strictly assigned according to the BearID methodology.

In [ ]:
from kfp import compiler, dsl, kubernetes
from kfp.dsl import ContainerSpec, Dataset, Input, OutputPath

# =====================================================================
# Decoupled Container Components (No packages_to_install allowed)
# =====================================================================

@dsl.container_component
def sync_s3_batch(s3_input_uri: str, raw_dataset: OutputPath(Dataset)) -> ContainerSpec:
    return ContainerSpec(
        image="ghcr.io/amlcoftherockies/kubeflow-bear-id/ingest:latest",
        command=["python3", "/app/sync_s3_batch.py"],
        args=["--s3-uri", s3_input_uri, "--output-path", raw_dataset]
    )

@dsl.container_component
def bearid_preprocessor(raw_dataset: Input[Dataset], aligned_tensors: OutputPath(Dataset)) -> ContainerSpec:
    return ContainerSpec(
        image="ghcr.io/amlcoftherockies/kubeflow-bear-id/preprocess:latest",
        command=["python3", "/app/align_and_tensorize.py"],
        args=["--input-dir", raw_dataset.path, "--output-dir", aligned_tensors]
    )

@dsl.container_component
def run_katib_ddp_sweep(experiment_name: str, namespace: str, optimal_params: OutputPath(Dataset)) -> ContainerSpec:
    return ContainerSpec(
        image="ghcr.io/amlcoftherockies/kubeflow-bear-id/classifier:latest", # Contains Katib SDK
        command=["python3", "/app/submit_katib.py"],
        args=["--name", experiment_name, "--namespace", namespace, "--out", optimal_params]
    )

@dsl.container_component
def bearsvm_fitter(
    aligned_tensors: Input[Dataset], 
    optimal_params: Input[Dataset], 
    mlflow_uri: str
) -> ContainerSpec:
    return ContainerSpec(
        image="ghcr.io/amlcoftherockies/kubeflow-bear-id/classifier:latest", # Contains MLflow/Sklearn
        command=["python3", "/app/fit_svm.py"],
        args=["--tensors", aligned_tensors.path, "--params", optimal_params.path, "--mlflow-uri", mlflow_uri]
    )

@dsl.container_component
def push_inference_graph(git_repo_url: str, git_branch: str, namespace: str) -> ContainerSpec:
    return ContainerSpec(
        image="ghcr.io/amlcoftherockies/kubeflow-bear-id/classifier:latest", # Contains GitPython/PyYAML
        command=["python3", "/app/push_gitops_graph.py"],
        args=["--repo-url", git_repo_url, "--branch", git_branch, "--namespace", namespace]
    )

# =====================================================================
# The Event-Driven DAG
# =====================================================================

@dsl.pipeline(name="bearid-ddp-inferencegraph-pipeline")
def bearid_pipeline(
    s3_input_uri: str,
    experiment_name: str = "bearid-resnet-tuning",
    namespace: str = "kubeflow-user",
    mlflow_uri: str = "http://mlflow.kubeflow.svc.cluster.local:5000",
    git_repo_url: str = "github.com/amlcoftherockies/kubeflow-infrastructure.git",
    git_branch: str = "main"
):
    sync = sync_s3_batch(s3_input_uri=s3_input_uri)
    
    # Dlib alignment is heavily CPU bound
    preprocess = bearid_preprocessor(raw_dataset=sync.outputs["raw_dataset"])
    preprocess.set_cpu_request('4').set_memory_request('8G')
    
    # Katib Launcher (Lightweight, orchestrates PyTorchJobs with GPUs under the hood)
    katib = run_katib_ddp_sweep(experiment_name=experiment_name, namespace=namespace)
    
    # The SVM fitter waits for Katib to finish, then pulls the winning model from MLflow.
    svm = bearsvm_fitter(
        aligned_tensors=preprocess.outputs["aligned_tensors"],
        optimal_params=katib.outputs["optimal_params"],
        mlflow_uri=mlflow_uri
    )
    svm.set_cpu_request('2').set_memory_request('4G')
    
    gitops = push_inference_graph(git_repo_url=git_repo_url, git_branch=git_branch, namespace=namespace)
    gitops.after(svm)
    
    kubernetes.use_secret_as_env(task=gitops, secret_name="github-auth-secret", secret_key_to_env={"token": "GIT_PAT"})

compiler.Compiler().compile(pipeline_func=bearid_pipeline, package_path="bearid_pipeline.yaml")
